# Phase 1 — Second Model Feasibility Check: Gemma-2-2B-it

## Purpose
Quick feasibility check before committing to the full pipeline on a second model
(addresses reviewer concern: "what if the null result is simply because 1.5B is
too small?"). Gemma-2-2B is close in size to Qwen2.5-1.5B-Instruct, so this
specifically avoids re-litigating the OOM/batch-size fights we had with 7B models,
while still testing whether the labeling pipeline and pivot rates transfer to a
different model family.

## What this notebook does
Runs the exact same 88-prompt set and proven-working generation config used for
the final Qwen2.5-1.5B dataset (Appendix A of the paper), on Gemma-2-2B-it instead.
Ends with a pivot-rate check — if pivot rates come back reasonable (15%+), it's
worth continuing to Phase 2 on this model. If not, stop here before investing
more time.

## Known Gemma-specific fix (already applied below)
Gemma-2's chat template does **not** support a `system` role message — passing
one raises a template error. We fold the system instructions into the single
user turn instead, which is the standard workaround.

## Before running
1. Session options → Accelerator → GPU T4 (single T4 is enough — Gemma-2-2B is
   close in size to Qwen2.5-1.5B, no OOM issues expected)
2. Gemma models require accepting Google's license on the model page first:
   https://huggingface.co/google/gemma-2-2b-it — click "Acknowledge license"
   (instant, unlike Llama's approval queue)
3. Add your `HF_TOKEN` as a Kaggle Secret (Add-ons → Secrets) if not already done

## 1. Install dependencies and verify GPU

In [1]:
!pip install -q transformers accelerate

import torch, json, os, re, time
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'No GPU — Session options -> Accelerator -> GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


## 2. Load tokenizer and model

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token, add_to_git_credential=False)

MODEL_NAME = 'google/gemma-2-2b-it'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # required for batched generation

print(f'Model loaded — {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters')

Loading google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded — 2.61B parameters


## 3. System prompt and prompt formatting

Using the exact system prompt that produced our final working Qwen dataset —
no few-shot examples (these caused severe problems earlier in this project:
82-93% false positive pivot rates from the model confusing example content
with the real question), no meta-cognitive additions (these also reduced
pivot rate when tried on Qwen).

**Gemma-specific:** folded into a single `user` turn since Gemma-2's chat
template does not support a separate `system` role.

In [3]:
SYSTEM_PROMPT = """You are a careful, honest reasoner. When solving a problem:
 Think through the problem step by step in plain prose, working toward an answer.

If, partway through, you realize your reasoning so far contains an actual error —
not because you were told to check, but because you have spotted something
concretely wrong — stop and correct it before continuing. If your reasoning is
already correct, continue straight to the final answer without pausing to
announce that you checked it.

Do not narrate a "checking," "pause," or "review" step unless you have genuinely
found something to fix. Do not use headers, bold text, or numbered steps to
structure your answer — just write continuous prose."""

def make_prompt(question: str) -> str:
    # Gemma-2 has no system role — fold system instructions into the user turn
    combined = (
        SYSTEM_PROMPT + "\n\n" + question +
        "\n\nRemember: commit to a first answer, then double-check it. Show your doubt explicitly."
    )
    messages = [{"role": "user", "content": combined}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Verify formatting works and looks correct
print(make_prompt("Test question")[:600])

<bos><start_of_turn>user
You are a careful, honest reasoner. When solving a problem:
 Think through the problem step by step in plain prose, working toward an answer.

If, partway through, you realize your reasoning so far contains an actual error —
not because you were told to check, but because you have spotted something
concretely wrong — stop and correct it before continuing. If your reasoning is
already correct, continue straight to the final answer without pausing to
announce that you checked it.

Do not narrate a "checking," "pause," or "review" step unless you have genuinely
found some


## 4. Prompt set

The full 88-prompt set from the paper (Appendix A) — same prompts used for the
final Qwen2.5-1.5B dataset, for a direct, apples-to-apples comparison.
10 misdirection + 40 math + 20 logic + 18 misdirection-expansion batch.

In [4]:
PROMPTS = [
    # ===== MISDIRECTION (10) =====
    {"id": "mis_01", "category": "misdirection", "text": "A rooster sits on a barn roof. It lays an egg. The wind is blowing east. Which direction does the egg roll?"},
    {"id": "mis_02", "category": "misdirection", "text": "How many animals of each type did Columbus take on the ark?"},
    {"id": "mis_03", "category": "misdirection", "text": "A helicopter crashes exactly on the border of Mexico and the US. In which country do they bury the survivors?"},
    {"id": "mis_04", "category": "misdirection", "text": "A doctor gives you 4 pills. Take one every 20 minutes. How long until they are all gone?"},
    {"id": "mis_05", "category": "misdirection", "text": "You are in a race. You overtake the last-place runner. What position are you now in?"},
    {"id": "mis_06", "category": "misdirection", "text": "There are 10 oranges in a bowl. You take away 4. How many oranges do you have?"},
    {"id": "mis_07", "category": "misdirection", "text": "A farmer has 20 cows. All but 7 are sold. How many cows remain?"},
    {"id": "mis_08", "category": "misdirection", "text": "If you only have one candle and walk into a freezing dark cabin with a wood stove, an oil lamp, and some newspaper — what do you light first?"},
    {"id": "mis_09", "category": "misdirection", "text": "A tennis ball and a racket together cost $51. The racket costs $50 more than the ball. How much does the ball cost?"},
    {"id": "mis_10", "category": "misdirection", "text": "Some months have 31 days, some have 30. How many have 28 days?"},

    # ===== MATH (40) =====
    {"id": "math_01", "category": "math", "text": "A car travels 30 miles at 30 mph, then 30 miles at 60 mph. Most people say the average speed is 45 mph. Are they right?"},
    {"id": "math_02", "category": "math", "text": "A price increases by 25% then decreases by 25%. A student says the final price equals the original. Check if this is correct."},
    {"id": "math_03", "category": "math", "text": "I have $200. I lose half, then I gain half of what I have left. My friend says I'm back to $200. Is my friend right?"},
    {"id": "math_04", "category": "math", "text": "A snail climbs 5 feet up a 10-foot well each day and slides back 3 feet each night. A student says it takes 5 days. Verify this."},
    {"id": "math_05", "category": "math", "text": "30% off then an extra 20% off. A shopper says that's 50% off total. Are they correct?"},
    {"id": "math_06", "category": "math", "text": "If 5 machines make 5 parts in 5 minutes, how long does 1 machine take to make 1 part? My first instinct is 1 minute — is that right?"},
    {"id": "math_07", "category": "math", "text": "There are 10 pairs of socks in a drawer — 5 black, 5 white. Lights are off. Someone says you need 3 draws to guarantee a matching pair. Check this."},
    {"id": "math_08", "category": "math", "text": "A rectangle's length increases by 20% and width decreases by 20%. Someone claims the area stays the same. Is that true?"},
    {"id": "math_09", "category": "math", "text": "Two trains 300 miles apart head toward each other at 80 mph and 70 mph. Someone says they meet in 2 hours. Verify this."},
    {"id": "math_10", "category": "math", "text": "You flip a coin 4 times and get heads every time. Someone claims the next flip is more likely to be tails. Are they right?"},
    {"id": "math_11", "category": "math", "text": "A clock shows 3:15. What is the angle between the hour and minute hands?"},
    {"id": "math_12", "category": "math", "text": "How many times does the digit 1 appear when you write all numbers from 1 to 100?"},
    {"id": "math_13", "category": "math", "text": "A ball is dropped from 10 feet. Each bounce reaches half the previous height. How far has it traveled after 3 bounces?"},
    {"id": "math_14", "category": "math", "text": "You have a 3-minute hourglass and a 5-minute hourglass. How do you time exactly 7 minutes?"},
    {"id": "math_15", "category": "math", "text": "If you fold a paper in half 7 times, how many layers do you have?"},
    {"id": "math_16", "category": "math", "text": "A ladder 10 feet long leans against a wall. The base is 6 feet from the wall. How high does it reach?"},
    {"id": "math_17", "category": "math", "text": "You roll two dice. What is the probability of getting a sum of 7?"},
    {"id": "math_18", "category": "math", "text": "A store sells 3 items for $1. How many can you buy for $10?"},
    {"id": "math_19", "category": "math", "text": "How many squares are on a standard 8x8 chessboard? Most people say 64."},
    {"id": "math_20", "category": "math", "text": "A palindrome number reads the same forwards and backwards. How many 3-digit palindromes exist?"},
    {"id": "math_21", "category": "math", "text": "If today is Wednesday, what day is it 100 days from now?"},
    {"id": "math_22", "category": "math", "text": "You have a 6-sided die. What is the expected value of a single roll?"},
    {"id": "math_23", "category": "math", "text": "A pizza is cut into 8 slices. You eat 3 slices. What percentage is left?"},
    {"id": "math_24", "category": "math", "text": "How many handshakes occur if 6 people each shake hands with every other person exactly once?"},
    {"id": "math_25", "category": "math", "text": "A number is divisible by both 4 and 6. Must it be divisible by 24?"},
    {"id": "math_26", "category": "math", "text": "You drive 1 mile at 30 mph. How fast must you drive the next mile to average 60 mph for both miles?"},
    {"id": "math_27", "category": "math", "text": "A box contains 4 red and 6 blue balls. You pick two without replacement. What is the probability both are red?"},
    {"id": "math_28", "category": "math", "text": "If 1/3 of a number is 4 more than 1/6 of the same number, what is the number?"},
    {"id": "math_29", "category": "math", "text": "A triangle has sides 3, 4, and 6. Is it a right triangle?"},
    {"id": "math_30", "category": "math", "text": "You invest $1000 at 10% annual interest compounded yearly. How much after 3 years? Someone says $1300."},
    {"id": "math_31", "category": "math", "text": "A sequence: 1, 1, 2, 3, 5, 8, 13... What is the ratio of consecutive terms approaching? Most people say exactly 2."},
    {"id": "math_32", "category": "math", "text": "If you have a 10x10x10 cube made of 1x1x1 smaller cubes, how many small cubes are on the surface? Most people say 600."},
    {"id": "math_33", "category": "math", "text": "A rope goes around the Earth's equator tightly. You add 1 meter to the rope's length. How high above the surface does it float uniformly? Most people say barely anything."},
    {"id": "math_34", "category": "math", "text": "How many zeros does 100 factorial (100!) end in? Someone says 20."},
    {"id": "math_35", "category": "math", "text": "You have a 1000-page book. How many times does the digit 1 appear in all page numbers?"},
    {"id": "math_36", "category": "math", "text": "A room has 30 people. Someone bets at least 2 share a birthday. Is this a good bet? Most say no since there are 365 days."},
    {"id": "math_37", "category": "math", "text": "You cut a Mobius strip down the middle lengthwise. How many pieces do you get? Most people say 2."},
    {"id": "math_38", "category": "math", "text": "0.999 repeating — is it equal to 1, less than 1, or greater than 1? Most people say less than 1."},
    {"id": "math_39", "category": "math", "text": "A bat costs $1 more than a ball. Together they cost $1.10. My friend says the ball costs 10 cents. Verify carefully."},
    {"id": "math_40", "category": "math", "text": "There are 23 people in a room. Someone bets two share a birthday. What are the actual odds? Most guess around 6%."},

    # ===== LOGIC / PROBABILITY (20) =====
    {"id": "logic_01", "category": "logic", "text": "You're on a game show. Three doors: one hides a car, two hide goats. You pick door 2. Host opens door 1 showing a goat. Should you switch to door 3?"},
    {"id": "logic_02", "category": "logic", "text": "A test is 95% accurate. A disease affects 1 in 1000 people. You test positive. Are you more likely to have the disease or not?"},
    {"id": "logic_03", "category": "logic", "text": "Mark is quiet, detail-oriented, and loves puzzles. Is it more likely he is a librarian or a farmer?"},
    {"id": "logic_04", "category": "logic", "text": "Two coins add up to 55 cents. One is not a nickel. What are the coins?"},
    {"id": "logic_05", "category": "logic", "text": "A father is 4 times as old as his son. In 20 years he will be twice as old. How old is the son now?"},
    {"id": "logic_06", "category": "logic", "text": "All squares are rectangles. Some rectangles are not squares. Can a rectangle be a square?"},
    {"id": "logic_07", "category": "logic", "text": "Three boxes: one has apples, one has oranges, one has both. All labels are wrong. You pick one fruit from the 'both' box. How do you label all three boxes correctly?"},
    {"id": "logic_08", "category": "logic", "text": "I have two children. At least one is a girl. What is the probability both are girls?"},
    {"id": "logic_09", "category": "logic", "text": "A man is found dead in a room. There is a puddle of water and broken glass on the floor. No windows, no other exits. What happened?"},
    {"id": "logic_10", "category": "logic", "text": "You have 12 balls, identical except one is slightly heavier. With a balance scale and only 3 weighings, how do you find it?"},
    {"id": "logic_11", "category": "logic", "text": "A monk climbs a mountain starting at 6am, reaches the top at 6pm, sleeps, then descends next day 6am to 6pm same path. Is there a point he passes at exactly the same time both days?"},
    {"id": "logic_12", "category": "logic", "text": "You have 3 light switches outside a room. One controls a bulb inside. You can only enter the room once. How do you find which switch controls the bulb?"},
    {"id": "logic_13", "category": "logic", "text": "A prisoner is told he will be hanged on an unexpected day next week. He reasons he cannot be hanged at all. Is his reasoning valid?"},
    {"id": "logic_14", "category": "logic", "text": "100 prisoners each have a number 1-100 on their back. Each can see others but not their own. Each must guess their own number simultaneously. Can they guarantee at least 1 correct guess?"},
    {"id": "logic_15", "category": "logic", "text": "You're in a room with two doors. One leads to freedom, one to death. Two guards: one always lies, one always tells the truth. You don't know which is which. You get one question. What do you ask?"},
    {"id": "logic_16", "category": "logic", "text": "There are 3 cards: one red both sides, one white both sides, one red one side white the other. You pick a card and see a red side. What is the probability the other side is also red?"},
    {"id": "logic_17", "category": "logic", "text": "Five houses in a row, each a different color, each resident drinks a different drink, smokes a different brand, owns a different pet. The Brit lives in the red house. Who owns the fish?"},
    {"id": "logic_18", "category": "logic", "text": "A father says to his son: 'I was your age when you were born.' The son is 20. How old is the father?"},
    {"id": "logic_19", "category": "logic", "text": "You have a 3-gallon and 5-gallon jug, no markings. You need exactly 4 gallons. Most people say it's impossible. Are they right?"},
    {"id": "logic_20", "category": "logic", "text": "A woman shoots her husband, then holds him underwater for 5 minutes. An hour later they go out to dinner. How?"},

    # ===== MISDIRECTION EXPANSION BATCH (18) =====
    {"id": "mis2_01", "category": "misdirection", "text": "A shepherd has 25 goats. All but 8 wander off. How many goats does he have left?"},
    {"id": "mis2_02", "category": "misdirection", "text": "You have 15 candies. You give away all but 5. How many do you have?"},
    {"id": "mis2_03", "category": "misdirection", "text": "A baker makes 40 loaves. All but 12 are sold. How many loaves are unsold?"},
    {"id": "mis2_04", "category": "misdirection", "text": "There are 8 fish in a tank. 3 of them drown. How many fish are left?"},
    {"id": "mis2_05", "category": "misdirection", "text": "A librarian has 60 books. All but 20 are checked out. How many are still on the shelf?"},
    {"id": "mis2_06", "category": "misdirection", "text": "You find a diamond in the desert and put it in your pocket, which has a hole in it. You keep walking for an hour. Do you still have the diamond?"},
    {"id": "mis2_07", "category": "misdirection", "text": "A man builds a house with four walls, each one facing south. A bear walks past the house. What color is the bear?"},
    {"id": "mis2_08", "category": "misdirection", "text": "A plane crashes exactly on the equator. In which hemisphere do they bury the survivors?"},
    {"id": "mis2_09", "category": "misdirection", "text": "You have 30 marbles. You lose all but 9. How many marbles remain?"},
    {"id": "mis2_10", "category": "misdirection", "text": "A grocer has 50 apples. All but 14 are rotten. How many good apples does he have?"},
    {"id": "mis2_11", "category": "misdirection", "text": "There are 6 eggs in a basket. You take away 4. How many eggs do you have?"},
    {"id": "mis2_12", "category": "misdirection", "text": "A man pushes his car to a hotel and, upon arriving, instantly loses everything he owns. What happened?"},
    {"id": "mis2_13", "category": "misdirection", "text": "How many months of the year have exactly 30 days?"},
    {"id": "mis2_14", "category": "misdirection", "text": "Before Mount Everest was discovered, what was the tallest mountain in the world?"},
    {"id": "mis2_15", "category": "misdirection", "text": "A truck driver is going the wrong way down a one-way street. A police officer sees him clearly but does not stop him. Why?"},
    {"id": "mis2_16", "category": "misdirection", "text": "A farmer had 12 cows. All but 5 died of a disease. How many cows does he have left?"},
    {"id": "mis2_17", "category": "misdirection", "text": "You have 3 apples and you take away 2. How many apples do you have?"},
    {"id": "mis2_18", "category": "misdirection", "text": "A clerk had 45 pencils. All but 13 broke. How many unbroken pencils remain?"},
]

print(f'Total prompts: {len(PROMPTS)}')
print(f'  Misdirection: {sum(1 for p in PROMPTS if p["category"] == "misdirection")}')
print(f'  Math:         {sum(1 for p in PROMPTS if p["category"] == "math")}')
print(f'  Logic:        {sum(1 for p in PROMPTS if p["category"] == "logic")}')

Total prompts: 88
  Misdirection: 28
  Math:         40
  Logic:        20


## 5. Sampling configuration

Same proven config as the final Qwen run: temperature 0.9, batch size 8
(Gemma-2-2B is close in size to Qwen2.5-1.5B, so no OOM expected on a single T4 —
if you do hit OOM, drop BATCH_SIZE to 4).

In [5]:
SAMPLES_PER_PROMPT = 15
TEMPERATURE = 0.9
TOP_P = 0.95
MAX_NEW_TOKENS = 400
BATCH_SIZE = 8   # drop to 4 if you hit CUDA OOM

TOTAL_GENERATIONS = len(PROMPTS) * SAMPLES_PER_PROMPT
print(f'Will generate {TOTAL_GENERATIONS} total outputs')
print(f'  ({len(PROMPTS)} prompts x {SAMPLES_PER_PROMPT} samples each)')

PROJECT_DIR = '/kaggle/working/pivot_research_gemma'
os.makedirs(PROJECT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(PROJECT_DIR, 'raw_outputs_gemma.jsonl')

# Clear any previous run of this notebook
if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)
    print('Cleared previous output file')

Will generate 1320 total outputs
  (88 prompts x 15 samples each)


## 6. Generation loop (batched)

In [6]:
def generate_batch(items):
    torch.manual_seed(items[0]['seed'])
    prompts = [make_prompt(item['prompt_text']) for item in items]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
            top_p=TOP_P, do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]

all_jobs = [
    {'prompt_id': p['id'], 'category': p['category'], 'prompt_text': p['text'],
     'sample_idx': s, 'seed': hash((p['id'], s)) % (2**31)}
    for p in PROMPTS for s in range(SAMPLES_PER_PROMPT)
]

print(f'Jobs to run: {len(all_jobs)}  |  Batch size: {BATCH_SIZE}')
start_time = time.time()

with open(OUTPUT_PATH, 'a') as fout:
    pbar = tqdm(total=len(all_jobs), desc='Generating (Gemma-2-2B)')
    for i in range(0, len(all_jobs), BATCH_SIZE):
        batch = all_jobs[i:i+BATCH_SIZE]
        try:
            gens = generate_batch(batch)
        except RuntimeError as e:
            if 'CUDA out of memory' in str(e):
                print(f'\nOOM — reduce BATCH_SIZE from {BATCH_SIZE} to {BATCH_SIZE//2} and rerun')
                break
            raise
        for job, gen in zip(batch, gens):
            fout.write(json.dumps({
                'prompt_id': job['prompt_id'], 'category': job['category'],
                'prompt_text': job['prompt_text'], 'sample_idx': job['sample_idx'],
                'generated_text': gen, 'full_text': job['prompt_text'] + '\n\n' + gen,
                'model_name': MODEL_NAME, 'temperature': TEMPERATURE, 'top_p': TOP_P,
                'seed': job['seed'], 'timestamp': datetime.now().isoformat(),
            }) + '\n')
        fout.flush()
        os.fsync(fout.fileno())
        pbar.update(len(batch))
    pbar.close()

print(f'\nDone in {(time.time()-start_time)/60:.1f} min')
print(f'Saved to: {OUTPUT_PATH}')

Jobs to run: 1320  |  Batch size: 8


Generating (Gemma-2-2B):   0%|          | 0/1320 [00:00<?, ?it/s]


Done in 47.5 min
Saved to: /kaggle/working/pivot_research_gemma/raw_outputs_gemma.jsonl


## 7. Pivot rate check — the feasibility decision point

Same tight regex used throughout this project (the loose version caused a 93%
false-positive rate on Qwen early on — this is the corrected, final version).

**Decision rule:**
- **15%+ overall pivot rate** → worth continuing to Phase 2 on this model
- **Below 15%** → before giving up, check the per-category breakdown; if
  misdirection specifically is reasonable (20%+), a targeted expansion batch
  (like `mis2_*` was for Qwen) may be enough to fix it
- **Below ~5% everywhere with incoherent sample outputs** → likely not worth
  pursuing this model; consider Mistral-7B or Llama-3.1-8B instead

In [7]:
PIVOT_PATTERNS = [
    r"\bwait,\s", r"\bwait—", r"\bactually,\s+(?:no|let|i|that|this|the|we|my|on)",
    r"\bhmm,\s", r"\bi made an error", r"\bi was wrong",
    r"\bthat('s| is) (wrong|incorrect|not right)", r"\blet me reconsider",
    r"\blet me recalculate", r"\blet me re-?read", r"\bon second thought",
    r"\bi need to correct", r"\bno,\s+wait", r"\bhold on",
    r"\bi('ve| have) made a mistake", r"\bactually,\s+i\b", r"\bactually,\s+no\b", r"\bactually,\s+that\b",
]
PIVOT_RE = re.compile("|".join(PIVOT_PATTERNS), re.IGNORECASE)

records = [json.loads(l) for l in open(OUTPUT_PATH)]
pivots = [r for r in records if PIVOT_RE.search(r['generated_text'])]

print(f'Total: {len(records)}  |  Pivots: {len(pivots)}  ({len(pivots)/len(records):.1%})')
print()
print('By category:')
for cat in ['misdirection', 'math', 'logic']:
    cat_all = [r for r in records if r['category'] == cat]
    cat_pivot = [r for r in pivots if r['category'] == cat]
    print(f"  {cat:<16} {len(cat_pivot):>3}/{len(cat_all)}  ({len(cat_pivot)/len(cat_all):>5.1%})")

FILTERED_PATH = os.path.join(PROJECT_DIR, 'pivot_candidates_gemma.jsonl')
with open(FILTERED_PATH, 'w') as f:
    for r in pivots:
        f.write(json.dumps(r) + '\n')
print(f'\nSaved pivot candidates to: {FILTERED_PATH}')

print()
print('--- Sample with pivot ---')
import random
if pivots:
    sample = random.choice(pivots)
    print(f"[{sample['prompt_id']} #{sample['sample_idx']}]")
    print(f"PROMPT: {sample['prompt_text']}")
    print(f"GEN:    {sample['generated_text'][:600]}")
else:
    print('No pivots found — check sample outputs below for coherence issues')
    print(records[0]['generated_text'][:600])

Total: 1320  |  Pivots: 111  (8.4%)

By category:
  misdirection      59/420  (14.0%)
  math              42/600  ( 7.0%)
  logic             10/300  ( 3.3%)

Saved pivot candidates to: /kaggle/working/pivot_research_gemma/pivot_candidates_gemma.jsonl

--- Sample with pivot ---
[mis_07 #13]
PROMPT: A farmer has 20 cows. All but 7 are sold. How many cows remain?
GEN:    If all but 7 cows are sold, that means 7 cows remain. This seems logical.  But wait, if all but 7 were sold, that would leave 7 cows, not the number of cows remaining. 



## 8. Next steps

**If pivot rate looks workable (15%+ overall):** proceed to Phase 2 (pivot
labeling) using the same regex and GROUND_TRUTH approach as the Qwen pipeline —
the keyword table will need new entries only if you add new prompts; the
existing 88-prompt keyword table (paper Appendix B) applies directly since
these are the same prompts with the same correct answers.

**If pivot rate is low overall but one category is workable:** consider a
targeted expansion batch for that category, following the same pattern used
for Qwen's `mis2_*` misdirection expansion.

**If pivot rate is low everywhere and outputs look incoherent:** this model
may not be a good fit at this prompt/system-prompt configuration. Worth a
quick sanity check of 2-3 sample outputs before concluding this — sometimes a
single system-prompt tweak (as happened repeatedly with Qwen) is all that's
needed. If outputs are coherent but genuinely rarely self-correct, that
itself may be worth noting as a cross-model observation.

Download `pivot_candidates_gemma.jsonl` from the Kaggle Output tab once
you've decided whether to continue.

In [9]:
# Quality check: how many of the 110 pivot candidates show genuine answer
# changes vs. stylistic restatement, and how many trail off unresolved?
# Run against pivot_candidates_gemma.jsonl -- no regeneration needed.

import json, random

records = [json.loads(l) for l in open('/kaggle/working/pivot_research_gemma/pivot_candidates_gemma.jsonl')]
print(f'Total pivot candidates: {len(records)}\n')

# Flag likely-unresolved endings (punting back to the user, trailing off)
unresolved_patterns = ['let me know if', 'would you like me to', 'try a different approach',
                        'i can try again', 'does that help']
unresolved = [r for r in records if any(p in r['generated_text'].lower() for p in unresolved_patterns)]
print(f'Records with likely unresolved/punting endings: {len(unresolved)} ({len(unresolved)/len(records):.1%})')

# Print 8 random samples for manual read -- this is the real diagnostic
print('\n' + '='*70)
print('8 random samples for manual review:')
print('='*70)
for r in random.sample(records, min(8, len(records))):
    print(f"\n[{r['prompt_id']} #{r['sample_idx']}] ({r['category']})")
    print(f"PROMPT: {r['prompt_text']}")
    print(f"GEN:    {r['generated_text'][:500]}")
    print('-'*70)

Total pivot candidates: 111

Records with likely unresolved/punting endings: 3 (2.7%)

8 random samples for manual review:

[logic_12 #3] (logic)
PROMPT: You have 3 light switches outside a room. One controls a bulb inside. You can only enter the room once. How do you find which switch controls the bulb?
GEN:    Okay, I can solve this!  Here's my first thought:  I'll flip the switches, one at a time, observing the light. The key to getting to the solution is understanding that I don't need to have all three switches on. That's the simplest explanation.

Wait, wait, I'm going to think about this more carefully.  What if the switches control the bulb on/off, not the brightness of it.  Could that be a confounding factor?

It sounds like a good idea to flip the switches, but what if the switches control the
----------------------------------------------------------------------

[mis2_03 #14] (misdirection)
PROMPT: A baker makes 40 loaves. All but 12 are sold. How many loaves are unsold?
GE